# Fintech A/B Testing: From Hypothesis to Product Decision

**Purpose:** evaluate whether a simplified virtual-card application flow improves application completion conversion without materially worsening guardrail metrics.

> **Important:** this notebook uses synthetic educational data only. Results are not evidence about a real fintech product. The data-generating process intentionally sets the treatment conversion probability above control; this is a practice exercise, not a real experiment.

## 1. Business question and experiment design

**Decision question:** should the product team roll out a simplified application flow?

- **Control (A):** current application flow.
- **Treatment (B):** simplified application flow.
- **Unit of randomization:** unique user; each user appears once.
- **Primary metric:** application completion conversion (completed / assigned users).
- **Guardrails:** application error rate and completion time among users who complete the application.
- **Significance level:** 5%, two-sided test.
- **Target power:** 80%.
- **Planning assumptions:** 30% baseline conversion and a +2 percentage-point minimum detectable effect (MDE).

The primary metric is defined for all randomized users, not only users who started or completed the flow. This keeps the denominator aligned with the randomized population.

In [ ]:
from __future__ import annotations

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy import stats
from statsmodels.stats.power import NormalIndPower
from statsmodels.stats.proportion import (
    proportion_effectsize,
    proportions_ztest,
    proportion_confint,
)

SEED = 20261026
ALPHA = 0.05
TARGET_POWER = 0.80
BASELINE_CONVERSION = 0.30
MDE_ABSOLUTE = 0.02

pd.set_option('display.float_format', lambda x: f'{x:,.4f}')
print('Libraries imported. Seed:', SEED)

## 2. Estimate the required sample size

For planning, compare a baseline conversion of 30% with 32% (an absolute lift of 2 percentage points). We use a two-sided two-proportion normal-approximation power calculation. The calculated sample size is a planning estimate; the final analysis still uses the observed synthetic data.

In [ ]:
p_control_plan = BASELINE_CONVERSION
p_treatment_plan = BASELINE_CONVERSION + MDE_ABSOLUTE

if not 0 < p_control_plan < 1 or not 0 < p_treatment_plan < 1:
    raise ValueError('Planning conversion rates must be strictly between 0 and 1.')

standardized_effect = proportion_effectsize(p_treatment_plan, p_control_plan)
power_analysis = NormalIndPower()
n_per_group = math.ceil(
    power_analysis.solve_power(
        effect_size=abs(standardized_effect),
        power=TARGET_POWER,
        alpha=ALPHA,
        ratio=1.0,
        alternative='two-sided',
    )
)

sample_size_plan = pd.DataFrame({
    'parameter': ['Control conversion', 'Treatment conversion', 'Absolute MDE', 'Alpha', 'Target power', 'Required users per group', 'Total required users'],
    'value': [p_control_plan, p_treatment_plan, MDE_ABSOLUTE, ALPHA, TARGET_POWER, n_per_group, 2 * n_per_group],
})
display(sample_size_plan)
print(f'Planning estimate: {n_per_group:,} users per group ({2*n_per_group:,} total).')

## 3. Generate reproducible synthetic experiment data

Each user is randomly assigned to A or B with a 50/50 allocation. The outcome probabilities are fixed in advance for this simulation (30% in A, 32% in B). Guardrail values are also synthetic. Completion time is only populated for users who complete the application; it must therefore be interpreted conditionally, alongside the unconditional conversion metric.

In [ ]:
rng = np.random.default_rng(SEED)
n_total = 2 * n_per_group

user_ids = np.arange(1, n_total + 1)
variants = np.array(['control', 'treatment'] * n_per_group)
rng.shuffle(variants)

# Synthetic data-generating assumptions, not observed product facts.
conversion_probability = np.where(variants == 'control', 0.30, 0.32)
error_probability = np.where(variants == 'control', 0.030, 0.032)
completed = rng.binomial(1, conversion_probability)
application_error = rng.binomial(1, error_probability)

# Completion time exists only for completed applications. Treatment is simulated to be faster.
completion_time_minutes = np.full(n_total, np.nan, dtype=float)
control_mask = (variants == 'control') & (completed == 1)
treatment_mask = (variants == 'treatment') & (completed == 1)
completion_time_minutes[control_mask] = np.clip(rng.normal(8.5, 2.0, control_mask.sum()), 1.0, None)
completion_time_minutes[treatment_mask] = np.clip(rng.normal(7.7, 1.9, treatment_mask.sum()), 1.0, None)

experiment = pd.DataFrame({
    'user_id': user_ids,
    'variant': variants,
    'application_completed': completed,
    'application_error': application_error,
    'completion_time_minutes': completion_time_minutes,
})

print('Dataset shape:', experiment.shape)
display(experiment.head())
display(experiment.groupby('variant').size().rename('users').to_frame())

## 4. Data quality checks

These assertions check the experiment dataset before analysis: required columns, unique user IDs, valid variants and binary outcomes, non-negative completion times, and missing completion times for users who did not complete. A passing check only establishes that the data satisfy these structural rules; it does not prove that the experiment design is free of all bias.

In [ ]:
required_columns = {
    'user_id', 'variant', 'application_completed',
    'application_error', 'completion_time_minutes'
}
assert required_columns.issubset(experiment.columns), 'Required columns are missing.'
assert experiment['user_id'].is_unique, 'Each randomized user must appear once.'
assert set(experiment['variant'].unique()) == {'control', 'treatment'}, 'Unexpected variant labels.'
assert experiment['application_completed'].isin([0, 1]).all(), 'Completion must be binary.'
assert experiment['application_error'].isin([0, 1]).all(), 'Error outcome must be binary.'
assert experiment.loc[experiment['application_completed'] == 0, 'completion_time_minutes'].isna().all(), 'Non-completers must not have a completion time.'
assert experiment.loc[experiment['application_completed'] == 1, 'completion_time_minutes'].notna().all(), 'Completers must have a completion time.'
assert (experiment.loc[experiment['application_completed'] == 1, 'completion_time_minutes'] > 0).all(), 'Completion times must be positive.'
assert experiment.isna().sum().to_dict()['user_id'] == 0, 'User IDs must not be missing.'

allocation = experiment['variant'].value_counts(normalize=True)
print('All structural data-quality assertions passed.')
print('Allocation share:')
display(allocation.rename('share').to_frame())

## 5. Define and calculate metrics

Conversion and error rates use all assigned users as the denominator. Completion time is calculated among completers only, so it is a conditional metric and can be affected by changes in who completes. Report the denominator alongside each rate.

In [ ]:
summary = experiment.groupby('variant').agg(
    users=('user_id', 'nunique'),
    completed=('application_completed', 'sum'),
    errors=('application_error', 'sum'),
    completers_with_time=('completion_time_minutes', 'count'),
    mean_completion_time_minutes=('completion_time_minutes', 'mean'),
    median_completion_time_minutes=('completion_time_minutes', 'median'),
)
summary['conversion_rate'] = summary['completed'] / summary['users']
summary['error_rate'] = summary['errors'] / summary['users']
summary['completion_rate_denominator'] = summary['users']
summary['time_metric_denominator'] = summary['completers_with_time']
display(summary)

control = experiment.loc[experiment['variant'] == 'control']
treatment = experiment.loc[experiment['variant'] == 'treatment']
n_control = len(control)
n_treatment = len(treatment)
x_control = int(control['application_completed'].sum())
x_treatment = int(treatment['application_completed'].sum())
p_control = x_control / n_control
p_treatment = x_treatment / n_treatment
absolute_lift = p_treatment - p_control
relative_lift = absolute_lift / p_control if p_control else np.nan

primary_summary = pd.DataFrame({
    'metric': ['Control conversion', 'Treatment conversion', 'Absolute lift (percentage points)', 'Relative lift'],
    'value': [p_control, p_treatment, absolute_lift, relative_lift],
})
display(primary_summary)

## 6. Visualize primary metric

The plot shows observed conversion in each arm. It is descriptive; statistical uncertainty is evaluated in the following sections.

In [ ]:
conversion_plot = summary.loc[['control', 'treatment'], 'conversion_rate']
ax = conversion_plot.plot(kind='bar', rot=0, figsize=(7, 4), title='Application completion conversion by variant')
ax.set_xlabel('Variant')
ax.set_ylabel('Conversion rate')
ax.set_ylim(0, min(1.0, max(0.4, conversion_plot.max() + 0.05)))
ax.grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## 7. Primary inference: two-proportion z-test and confidence intervals

The null hypothesis is that control and treatment conversion are equal. The alternative is two-sided: the rates differ. We report a pooled two-proportion z-test and separate Wilson confidence intervals for each arm. For the difference, we use a normal-approximation interval; the bootstrap interval below provides a complementary estimate.

In [ ]:
H0 = 'p_treatment = p_control'
H1 = 'p_treatment != p_control'

z_stat, p_value_z = proportions_ztest(
    count=np.array([x_treatment, x_control]),
    nobs=np.array([n_treatment, n_control]),
    alternative='two-sided',
)

ci_control = proportion_confint(x_control, n_control, alpha=ALPHA, method='wilson')
ci_treatment = proportion_confint(x_treatment, n_treatment, alpha=ALPHA, method='wilson')
se_difference = math.sqrt(
    p_treatment * (1 - p_treatment) / n_treatment
    + p_control * (1 - p_control) / n_control
)
ci_difference = (
    absolute_lift - stats.norm.ppf(1 - ALPHA / 2) * se_difference,
    absolute_lift + stats.norm.ppf(1 - ALPHA / 2) * se_difference,
)

inference = pd.DataFrame({
    'quantity': ['Control conversion (Wilson 95% CI)', 'Treatment conversion (Wilson 95% CI)', 'Treatment - control (95% normal CI)', 'Two-proportion z statistic', 'Two-sided p-value'],
    'result': [
        f'{p_control:.4f} [{ci_control[0]:.4f}, {ci_control[1]:.4f}]',
        f'{p_treatment:.4f} [{ci_treatment[0]:.4f}, {ci_treatment[1]:.4f}]',
        f'{absolute_lift:.4f} [{ci_difference[0]:.4f}, {ci_difference[1]:.4f}]',
        f'{z_stat:.4f}',
        f'{p_value_z:.6g}',
    ]
})
display(inference)
print('Reject H0 at alpha=0.05:', p_value_z < ALPHA)

## 8. Cross-check with Pearson chi-square test

For a 2×2 table of variant by completion, Pearson's chi-square test provides a second test of association. With two groups and a binary outcome, it should generally agree closely with the two-proportion z-test for this sample size. This is a cross-check, not an independent replication of the experiment.

In [ ]:
contingency = pd.crosstab(experiment['variant'], experiment['application_completed']).reindex(
    index=['control', 'treatment'], columns=[0, 1], fill_value=0
)
chi2_stat, chi2_p_value, degrees_freedom, expected = stats.chi2_contingency(contingency, correction=False)
print('Contingency table (columns: not completed=0, completed=1):')
display(contingency)
print(f'Pearson chi-square={chi2_stat:.4f}, df={degrees_freedom}, p-value={chi2_p_value:.6g}')
print('Z-test and chi-square p-values close:', np.isclose(p_value_z, chi2_p_value, rtol=0.03, atol=1e-12))

## 9. Bootstrap confidence interval for absolute conversion lift

We resample users with replacement within each arm and calculate treatment minus control conversion for each replicate. Stratifying the bootstrap by arm preserves the observed arm sizes. The interval quantifies sampling uncertainty under this dataset; it does not address bias in the design or make synthetic data representative of real customers.

In [ ]:
N_BOOTSTRAP = 5000
bootstrap_rng = np.random.default_rng(SEED + 1)
control_outcomes = control['application_completed'].to_numpy()
treatment_outcomes = treatment['application_completed'].to_numpy()
bootstrap_lifts = np.empty(N_BOOTSTRAP)

for i in range(N_BOOTSTRAP):
    sampled_control = bootstrap_rng.choice(control_outcomes, size=n_control, replace=True)
    sampled_treatment = bootstrap_rng.choice(treatment_outcomes, size=n_treatment, replace=True)
    bootstrap_lifts[i] = sampled_treatment.mean() - sampled_control.mean()

bootstrap_ci = tuple(np.quantile(bootstrap_lifts, [ALPHA / 2, 1 - ALPHA / 2]))
print(f'Observed absolute lift: {absolute_lift:.4f} ({absolute_lift*100:.2f} percentage points)')
print(f'Bootstrap 95% percentile CI: [{bootstrap_ci[0]:.4f}, {bootstrap_ci[1]:.4f}]')

plt.figure(figsize=(8, 4))
plt.hist(bootstrap_lifts, bins=45)
plt.axvline(0, linestyle='--', label='No effect')
plt.axvline(absolute_lift, linestyle='-', label='Observed lift')
plt.xlabel('Treatment - control conversion')
plt.ylabel('Bootstrap replicates')
plt.title('Bootstrap distribution of absolute conversion lift')
plt.legend()
plt.tight_layout()
plt.show()

## 10. Guardrail analysis

Guardrails help identify potential harm that a primary-metric improvement could hide. Here, we compare application error rates using a two-proportion test and completion time among completers using Welch's t-test. Because completion time is conditional on completion, interpret it cautiously. These are secondary checks; if this were a real experiment, the guardrail thresholds and multiplicity approach should be specified before looking at results.

In [ ]:
control_errors = int(control['application_error'].sum())
treatment_errors = int(treatment['application_error'].sum())
error_z, error_p = proportions_ztest(
    count=np.array([treatment_errors, control_errors]),
    nobs=np.array([n_treatment, n_control]),
    alternative='two-sided',
)
error_rate_control = control_errors / n_control
error_rate_treatment = treatment_errors / n_treatment
error_rate_difference = error_rate_treatment - error_rate_control

control_times = control['completion_time_minutes'].dropna().to_numpy()
treatment_times = treatment['completion_time_minutes'].dropna().to_numpy()
time_t, time_p = stats.ttest_ind(treatment_times, control_times, equal_var=False, alternative='two-sided')
time_difference = treatment_times.mean() - control_times.mean()
time_se = math.sqrt(treatment_times.var(ddof=1)/len(treatment_times) + control_times.var(ddof=1)/len(control_times))
time_df_num = (treatment_times.var(ddof=1)/len(treatment_times) + control_times.var(ddof=1)/len(control_times))**2
time_df_den = (treatment_times.var(ddof=1)/len(treatment_times))**2/(len(treatment_times)-1) + (control_times.var(ddof=1)/len(control_times))**2/(len(control_times)-1)
time_df = time_df_num / time_df_den
time_critical = stats.t.ppf(1 - ALPHA/2, df=time_df)
time_ci = (time_difference - time_critical*time_se, time_difference + time_critical*time_se)

guardrails = pd.DataFrame({
    'metric': ['Application error rate', 'Mean completion time among completers'],
    'control': [error_rate_control, control_times.mean()],
    'treatment': [error_rate_treatment, treatment_times.mean()],
    'treatment_minus_control': [error_rate_difference, time_difference],
    'p_value': [error_p, time_p],
    'interpretation': [
        'Positive difference means more errors in treatment',
        'Negative difference means faster completion among completers',
    ],
})
display(guardrails)
print(f'Completion-time difference 95% Welch CI: [{time_ci[0]:.3f}, {time_ci[1]:.3f}] minutes')

## 11. Make a product recommendation using explicit rules

For this exercise, the illustrative decision rule is:

1. Primary metric is statistically significant at 5%.
2. The observed primary effect is positive.
3. The treatment error-rate increase is not more than **1 percentage point** (an illustrative guardrail threshold).

These rules are pedagogical assumptions, not universal policy. A real rollout decision would also consider practical significance, pre-registered metrics, instrumentation, segment checks, experiment integrity, risk/compliance review, and the cost of the change.

In [ ]:
MAX_ACCEPTABLE_ERROR_RATE_INCREASE = 0.01  # illustrative absolute threshold
primary_success = (p_value_z < ALPHA) and (absolute_lift > 0)
error_guardrail_pass = error_rate_difference <= MAX_ACCEPTABLE_ERROR_RATE_INCREASE

if primary_success and error_guardrail_pass:
    recommendation = 'CONSIDER ROLLOUT: primary metric passes the illustrative rule and error guardrail is within threshold.'
elif not primary_success:
    recommendation = 'DO NOT ROLL OUT BASED ON THIS RESULT: primary metric did not show a statistically significant positive effect.'
else:
    recommendation = 'DO NOT ROLL OUT YET: the illustrative error-rate guardrail threshold was exceeded.'

print('Primary metric statistically significant and positive:', primary_success)
print(f'Error-rate difference: {error_rate_difference:.4f}; allowed increase: {MAX_ACCEPTABLE_ERROR_RATE_INCREASE:.4f}')
print('Error guardrail passes:', error_guardrail_pass)
print('\nRecommendation:', recommendation)
print('\nReminder: this recommendation applies only to the synthetic simulation and must not be presented as a real product finding.')

## 12. Limitations and what a real experiment would require

- The dataset is synthetic and the treatment effect is built into the data generator.
- The analysis assumes valid randomization, one row per user, consistent instrumentation, and no interference between users.
- The sample-size calculation uses planning assumptions; it does not guarantee a significant result.
- The analysis uses one primary metric. Secondary metrics should be treated carefully to avoid overinterpreting multiple tests.
- Completion time is conditional on completion and may reflect a changing mix of users.
- This notebook does not include sequential monitoring, sample-ratio-mismatch diagnostics, segment-level analyses, or a production rollout framework.

Before using this workflow on a real experiment, agree on the hypothesis, primary metric, MDE, sample size, run duration, exclusion rules, guardrails, and decision criteria before reading outcomes.

## 13. Reproducibility

The random seed is fixed in the configuration cell. Restart the kernel and run all cells from top to bottom. The notebook intentionally stores no precomputed output; results should be generated by executing the code. Use `requirements.txt` for the main dependencies.